# PixelOrbit: Cross-Sensor Lunar Image Registration (ISRO PS 26166)
### Autonomous Deep Neural Registration Pipeline for Chandrayaan-2 OHRC & TMC-2
**Hardware Accelerator**: Nvidia T4 GPU (16 GB VRAM · Free on Google Colab)

This notebook launches the full interactive PixelOrbit prototype with:
- **RoMa-v2 Dense Transformer** (DINOv2 ViT-L/14 backbone with 16 GB VRAM acceleration)
- **LoFTR** (Detector-Free Feature Matcher)
- **LightGlue + DISK** (Adaptive Neural Matcher)
- **SIFT & ORB** (Classical Baselines)
- **CNSFM** (Crater Network Structure from Motion)
- **Fourier-Mellin Phase Correlation** (Scale-space 20:1 normalization)
- **Huber Sub-Pixel Homography Optimization** & **Laplacian Pyramid Fusion**
- **Interactive 3D Photometric Lunar-Lambert Terrain Visualizer**


## Step 1: Verify Nvidia GPU Hardware Acceleration
Make sure you have enabled a GPU under **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!nvidia-smi

## Step 2: Clone & Update PixelOrbit Repository

In [ ]:
import os
if not os.path.exists("PixelOrbit"):
    !git clone https://github.com/Helium7707/PixelOrbit.git
%cd PixelOrbit
!git pull origin main

## Step 3: Install Required Dependencies & Download Tunneling Utility

In [ ]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
!pip install -q pyngrok
# Download standalone Cloudflare Tunnel binary (100% free, zero token/account needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

## Step 4: Launch Streamlit App with Instant Cloudflare Tunnel
Run this cell to start the server and get a secure, free public URL to interact with the full GPU pipeline.

In [ ]:
import subprocess
import time
import re

# Terminate any previous instances
!pkill -f streamlit || true
!pkill -f cloudflared || true

print("[1/3] Starting PixelOrbit Streamlit application in background...")
st_cmd = ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true", "--server.enableCORS", "false"]
st_proc = subprocess.Popen(st_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

print("[2/3] Establishing Cloudflare secure public tunnel...")
cf_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for _ in range(40):
    line = cf_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        break

print("[3/3] Ready!")
if tunnel_url:
    print("
" + "="*72)
    print("  PIXELORBIT IS LIVE ON GOOGLE COLAB (NVIDIA GPU ACCELERATED)!")
    print(f"  Public URL: {tunnel_url}")
    print("="*72 + "
")
else:
    print("Tunnel started. If URL did not display above, use the Localtunnel or Ngrok cell below.")


## Alternative Tunnel Options (Optional)
Use these if Cloudflare is restricted on your network.

In [ ]:
# --- OPTION B: Localtunnel ---
# import urllib.request
# print("Your IP password is:", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip())
# !npx localtunnel --port 8501

# --- OPTION C: Ngrok (requires auth token from ngrok.com) ---
# from pyngrok import ngrok
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")
# public_url = ngrok.connect(8501)
# print("ngrok URL:", public_url)
